# MercadoTico 360 · Requisitos del Caso 2 (CouchDB)

Ejecute las celdas **en orden**. Cada sección corresponde a un requisito de la investigación y termina con su evidencia en pantalla. Al final hay un resumen de cumplimiento.

> Requiere haber cargado antes los datos con `Simulacion.ipynb`.

| Requisito | Sección | Evidencia de la demostración |
|---|---|---|
| 1 y 2. Diseño y atributos por categoría | Documentos reales | Visualización de documentos y diseño |
| 3. Índices para 3+ patrones | Índices | Utilidad de cada índice |
| 4. Categoría + precio + atributo | Búsquedas | Pruebas **antes y después** de los índices |
| 5. Pedidos de un cliente y cambio de estado | Pedidos | Pedido anidado, estado actualizado |
| 6. Dos o más agregaciones | MapReduce | Ventas por categoría, ticket promedio, top productos |
| 7. Evolución del esquema | Evolución | Nueva categoría y atributo, validación |

In [3]:
import json
import os
from getpass import getpass

import requests

try:                                   # Opcional: lee un archivo .env si existe
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


def leer_config(nombre, pregunta, defecto=None, secreto=False):
    """Toma el valor de una variable de entorno; si no existe, lo pregunta."""
    valor = os.getenv(nombre)
    if valor:
        return valor.strip()
    if secreto:
        return getpass(f"{pregunta}: ").strip()
    sufijo = f" [{defecto}]" if defecto else ""
    return input(f"{pregunta}{sufijo}: ").strip() or defecto


COUCHDB_HOST  = leer_config("COUCHDB_HOST", "Host/IP de CouchDB", "localhost")
COUCHDB_PORT  = leer_config("COUCHDB_PORT", "Puerto de CouchDB", "5984")
COUCHDB_USER  = leer_config("COUCHDB_USER", "Usuario de CouchDB", "admin")
COUCHDB_PASS  = leer_config("COUCHDB_PASS", "Contraseña de CouchDB", secreto=True)
DATABASE_NAME = leer_config("DATABASE_NAME", "Nombre de la base de datos", "mercadotico360")

DB_URL = f"http://{COUCHDB_HOST}:{COUCHDB_PORT}/{DATABASE_NAME}"

SESSION = requests.Session()
SESSION.auth = (COUCHDB_USER, COUCHDB_PASS)

## Funciones de apoyo
Todas las consultas usan `buscar()`, que imprime siempre las mismas métricas (resultados, documentos examinados y tiempo) para poder comparar.

In [4]:
RESULTADOS = {}      # resumen de cumplimiento (se imprime al final)


def registrar(requisito, ok, detalle=""):
    RESULTADOS[requisito] = (ok, detalle)
    print(f"  [{'OK' if ok else 'FALLO'}] {requisito}  {detalle}")


def buscar(selector, use_index=None, limit=100000, titulo=""):
    """Ejecuta una consulta Mango (_find) e imprime sus métricas de rendimiento."""
    payload = {"selector": selector, "limit": limit, "execution_stats": True}
    if use_index:
        payload["use_index"] = use_index
    data = SESSION.post(f"{DB_URL}/_find", json=payload, timeout=300).json()
    stats = data.get("execution_stats", {})
    print(f"{titulo}: {len(data.get('docs', []))} resultados | "
          f"docs examinados: {stats.get('total_docs_examined')} | "
          f"{stats.get('execution_time_ms')} ms")
    if data.get("warning"):
        print("   aviso de CouchDB:", data["warning"].split(".")[0])
    return data


def guardar_design_doc(ddoc):
    """Crea o actualiza un design document (vistas, update handlers, validaciones)."""
    url = f"{DB_URL}/{ddoc['_id']}"
    actual = SESSION.get(url, timeout=10)
    if actual.status_code == 200:
        ddoc["_rev"] = actual.json()["_rev"]      # necesario para actualizar
    SESSION.put(url, json=ddoc, timeout=60).raise_for_status()


def verificar_datos():
    """Comprueba que existan productos, clientes y pedidos cargados."""
    for tipo in ("producto", "cliente", "pedido"):
        r = SESSION.post(f"{DB_URL}/_find", timeout=120,
                         json={"selector": {"tipo_doc": tipo}, "limit": 1, "fields": ["_id"]}).json()
        if not r.get("docs"):
            raise RuntimeError(f"No hay documentos '{tipo}'. Ejecute primero Simulacion.ipynb.")
    print("[OK] La base de datos contiene productos, clientes y pedidos.")


verificar_datos()

[OK] La base de datos contiene productos, clientes y pedidos.


## Requisitos 1 y 2 · Documentos reales y atributos variables
**Qué se embebe y qué se referencia**
- `producto`: documento propio; sus `atributos` van **embebidos** y cambian según la categoría.
- `pedido`: las `lineas_detalle` van **embebidas** (con *snapshot* de nombre, categoría y precio); `cliente_id` y `producto_id` son **referencias**.
- `cliente`: documento propio, para que sus datos cambien sin tocar los pedidos.

In [5]:
def mostrar_productos_variables(categorias=("Ropa", "Tecnologia", "Artesanias")):
    """Muestra un producto de cada categoría: los atributos son distintos en cada una."""
    for cat in categorias:
        r = SESSION.post(f"{DB_URL}/_find", timeout=60,
                         json={"selector": {"tipo_doc": "producto", "categoria": cat}, "limit": 1}).json()
        if r.get("docs"):
            p = r["docs"][0]
            print(f"{p['producto_id']} | {p['categoria']} | atributos: {json.dumps(p['atributos'], ensure_ascii=False)}")


def mostrar_pedido_anidado(pedido_id="PED-000001"):
    """Muestra un pedido completo con sus líneas de detalle anidadas."""
    pedido = SESSION.get(f"{DB_URL}/{pedido_id}", timeout=10).json()
    print(json.dumps(pedido, indent=2, ensure_ascii=False))
    return pedido


print("--- Productos con atributos distintos por categoría (Req. 2) ---")
mostrar_productos_variables()
print("\n--- Pedido con estructura anidada (Req. 1) ---")
pedido_demo = mostrar_pedido_anidado()
registrar("Req 1-2: diseño y atributos variables", "lineas_detalle" in pedido_demo)

--- Productos con atributos distintos por categoría (Req. 2) ---
PROD-000001 | Ropa | atributos: {"talla": "S", "color": "Azul", "material": "Poliéster"}
PROD-000017 | Tecnologia | atributos: {"marca": "TechBrand-35", "ram_gb": 16, "almacenamiento_gb": 2048}
PROD-000008 | Artesanias | atributos: {"material": "Textil", "tecnica": "Torno", "region_origen": "San Vicente de Nicoya"}

--- Pedido con estructura anidada (Req. 1) ---
{
  "_id": "PED-000001",
  "_rev": "1-c65f7a5c47d98867e9ccf75bec847da2",
  "tipo_doc": "pedido",
  "pedido_id": "PED-000001",
  "cliente_id": "CLI-004637",
  "fecha_pedido": "2026-07-14",
  "lineas_detalle": [
    {
      "producto_id": "PROD-009498",
      "nombre_snapshot": "Alimento para mascota 9498",
      "categoria_snapshot": "Mascotas",
      "cantidad": 2,
      "precio_unitario": 626.5,
      "subtotal": 1253.0
    },
    {
      "producto_id": "PROD-002258",
      "nombre_snapshot": "Café 2258",
      "categoria_snapshot": "Alimentos",
      "cantidad":

## Requisito 3 · Índices para patrones de consulta frecuentes

| Índice | Campos | Patrón de consulta que acelera |
|---|---|---|
| `idx_product_category` | `tipo_doc`, `categoria` | Listar productos de una categoría (navegación del catálogo) |
| `idx_product_price` | `tipo_doc`, `precio` | Buscar productos dentro de un rango de precios |
| `idx_technology_ram` | `tipo_doc`, `categoria`, `atributos.ram_gb` | Filtrar por un atributo propio de una categoría (RAM en Tecnologia) |
| `idx_pedidos_cliente` | `tipo_doc`, `cliente_id` | Recuperar el historial de pedidos de un cliente (Req. 5) |

Mango usa **un solo índice por consulta**, así que los demás filtros se aplican en memoria sobre los documentos que ese índice devuelve.

In [6]:
INDICES = {
    "idx_product_category": ["tipo_doc", "categoria"],
    "idx_product_price": ["tipo_doc", "precio"],
    "idx_technology_ram": ["tipo_doc", "categoria", "atributos.ram_gb"],
    "idx_pedidos_cliente": ["tipo_doc", "cliente_id"],
}


def crear_indice(nombre):
    payload = {"index": {"fields": INDICES[nombre]}, "name": nombre, "type": "json"}
    res = SESSION.post(f"{DB_URL}/_index", json=payload, timeout=300).json()
    print(f"  {nombre}: {res.get('result')}")          # 'created' o 'exists'


def crear_indices():
    for nombre in INDICES:
        crear_indice(nombre)


def eliminar_indice(nombre):
    """Elimina un índice por su nombre (se usa para la prueba 'antes de índices')."""
    for idx in SESSION.get(f"{DB_URL}/_index", timeout=30).json().get("indexes", []):
        if idx["name"] == nombre and idx["type"] == "json":
            SESSION.delete(f"{DB_URL}/_index/{idx['ddoc']}/json/{nombre}", timeout=60)
            print(f"  {nombre}: eliminado")


print("Creando índices (puede tardar unos segundos)...")
crear_indices()
registrar("Req 3: índices", True, f"{len(INDICES)} índices")

Creando índices (puede tardar unos segundos)...
  idx_product_category: created
  idx_product_price: created
  idx_technology_ram: created
  idx_pedidos_cliente: created
  [OK] Req 3: índices  4 índices


## Requisito 4 · Búsquedas por categoría, rango de precio y atributo específico
Primero se mide la misma consulta **sin índices** y luego **con índice**, que es la evidencia pedida ("pruebas antes y después de crear índices").

In [7]:
CONSULTA_CATEGORIA_PRECIO = {
    "tipo_doc": "producto",
    "categoria": "Tecnologia",
    "precio": {"$gte": 100, "$lte": 300},
}


def comparar_antes_despues():
    """Mide la consulta categoría + precio sin índices y con índice."""
    print("ANTES: se eliminan los índices de productos")
    for nombre in ("idx_product_category", "idx_product_price", "idx_technology_ram"):
        eliminar_indice(nombre)
    antes = buscar(CONSULTA_CATEGORIA_PRECIO, titulo="SIN índices")

    print("\nDESPUÉS: se vuelven a crear")
    crear_indices()
    despues = buscar(CONSULTA_CATEGORIA_PRECIO, use_index="idx_product_price", titulo="CON idx_product_price")

    ea, ed = antes.get("execution_stats", {}), despues.get("execution_stats", {})
    mejora = ea.get("total_docs_examined", 0) / max(ed.get("total_docs_examined", 1), 1)
    print(f"\nDocumentos examinados: {ea.get('total_docs_examined')} -> {ed.get('total_docs_examined')} "
          f"({mejora:.1f}x menos)")
    ok = len(antes.get("docs", [])) == len(despues.get("docs", [])) and mejora > 1
    registrar("Req 4: antes/después de índices", ok, f"{mejora:.1f}x menos documentos examinados")


def buscar_categoria_precio_atributo(ram_gb=16, precio_min=50, precio_max=1500):
    """Tecnologia + rango de precio + atributo específico (RAM)."""
    selector = {
        "tipo_doc": "producto",
        "categoria": "Tecnologia",
        "atributos.ram_gb": ram_gb,
        "precio": {"$gte": precio_min, "$lte": precio_max},
    }
    data = buscar(selector, use_index="idx_technology_ram", titulo=f"Tecnologia, {ram_gb} GB RAM, ${precio_min}-${precio_max}")
    if data.get("docs"):
        ejemplo = data["docs"][0]
        print(f"   ejemplo: {ejemplo['nombre']} | ${ejemplo['precio']} | {ejemplo['atributos']}")
    registrar("Req 4: categoría + precio + atributo", bool(data.get("docs")), f"{len(data.get('docs', []))} productos")


comparar_antes_despues()
print()
buscar_categoria_precio_atributo()

ANTES: se eliminan los índices de productos
  idx_product_category: eliminado
  idx_product_price: eliminado
  idx_technology_ram: eliminado
SIN índices: 420 resultados | docs examinados: 85001 | 28076.0 ms
   aviso de CouchDB: No matching index found, create an index to optimize query time

DESPUÉS: se vuelven a crear
  idx_product_category: created
  idx_product_price: created
  idx_technology_ram: created
  idx_pedidos_cliente: exists
CON idx_product_price: 420 resultados | docs examinados: 3116 | 810.0 ms
   aviso de CouchDB: _design/idx_product_price was not used because it does not contain a valid index for this query

Documentos examinados: 85001 -> 3116 (27.3x menos)
  [OK] Req 4: antes/después de índices  27.3x menos documentos examinados

Tecnologia, 16 GB RAM, $50-$1500: 611 resultados | docs examinados: 628 | 179.0 ms
   aviso de CouchDB: _design/idx_technology_ram was not used because it does not contain a valid index for this query
   ejemplo: Laptop 17 | $373.71 | {'marc

## Requisito 5 · Pedidos de un cliente y cambio de estado
Para actualizar solo el estado se usa un *update handler* de CouchDB: el cliente envía únicamente `{"estado": ...}` y el servidor aplica el cambio, sin que el código tenga que reenviar el pedido completo.
(CouchDB internamente guarda una nueva revisión del documento.)

In [8]:
def instalar_update_handler():
    guardar_design_doc({
        "_id": "_design/pedidos",
        "updates": {
            "cambiar_estado": (
                "function (doc, req) {"
                "  if (!doc) { return [null, 'pedido no existe']; }"
                "  doc.estado = JSON.parse(req.body).estado;"
                "  return [doc, 'ok'];"
                "}"
            )
        },
    })


def cambiar_estado_pedido(pedido_id, nuevo_estado):
    url = f"{DB_URL}/_design/pedidos/_update/cambiar_estado/{pedido_id}"
    return SESSION.put(url, json={"estado": nuevo_estado}, timeout=30)


def recuperar_y_actualizar_pedido(cliente_id="CLI-000001", nuevo_estado="completado"):
    # 1. Recuperar los pedidos completos del cliente
    data = buscar({"tipo_doc": "pedido", "cliente_id": cliente_id},
                  use_index="idx_pedidos_cliente", limit=1000,
                  titulo=f"Pedidos de {cliente_id}")
    pedidos = data.get("docs", [])
    if not pedidos:
        registrar("Req 5: pedidos y actualización", False, "el cliente no tiene pedidos")
        return

    # 2. Actualizar el estado de uno (solo se envía el campo 'estado')
    pedido = next((p for p in pedidos if p["estado"] != nuevo_estado), pedidos[0])   # uno que sí cambie
    anterior = pedido["estado"]
    instalar_update_handler()
    res = cambiar_estado_pedido(pedido["_id"], nuevo_estado)

    # 3. Comprobar el cambio
    actualizado = SESSION.get(f"{DB_URL}/{pedido['_id']}", timeout=10).json()
    print(f"\n{pedido['pedido_id']}: '{anterior}' -> '{actualizado['estado']}'  (HTTP {res.status_code})")
    print(f"Líneas de detalle conservadas: {len(actualizado['lineas_detalle'])} (sin cambios)")
    registrar("Req 5: pedidos y actualización", actualizado["estado"] == nuevo_estado,
              f"{len(pedidos)} pedidos recuperados")


recuperar_y_actualizar_pedido("CLI-000001")

Pedidos de CLI-000001: 11 resultados | docs examinados: 11 | 9881.0 ms
   aviso de CouchDB: _design/idx_pedidos_cliente was not used because it does not contain a valid index for this query

PED-000215: 'cancelado' -> 'completado'  (HTTP 201)
Líneas de detalle conservadas: 1 (sin cambios)
  [OK] Req 5: pedidos y actualización  11 pedidos recuperados


## Requisito 6 · Agregaciones con MapReduce
Tres vistas sobre los pedidos (se excluyen los cancelados en ventas y productos vendidos):

| Vista | Qué responde | Reduce |
|---|---|---|
| `ventas_por_categoria` | Ventas totales por categoría (usa el *snapshot* de cada línea) | `_sum` |
| `ticket_por_estado` | Cantidad, suma, mínimo y máximo de pedidos → ticket promedio | `_stats` |
| `productos_mas_vendidos` | Unidades vendidas por producto | `_sum` |

La primera consulta construye la vista y puede tardar un poco.

In [9]:
def crear_vistas_agregacion():
    guardar_design_doc({
        "_id": "_design/agregaciones",
        "views": {
            "ventas_por_categoria": {
                "map": ("function (doc) { if (doc.tipo_doc === 'pedido' && doc.estado !== 'cancelado') {"
                        " doc.lineas_detalle.forEach(function (l) { emit(l.categoria_snapshot, l.subtotal); }); } }"),
                "reduce": "_sum",
            },
            "ticket_por_estado": {
                "map": "function (doc) { if (doc.tipo_doc === 'pedido') { emit(doc.estado, doc.monto_total); } }",
                "reduce": "_stats",
            },
            "productos_mas_vendidos": {
                "map": ("function (doc) { if (doc.tipo_doc === 'pedido' && doc.estado !== 'cancelado') {"
                        " doc.lineas_detalle.forEach(function (l) { emit(l.producto_id, l.cantidad); }); } }"),
                "reduce": "_sum",
            },
        },
    })


def consultar_vista(nombre):
    url = f"{DB_URL}/_design/agregaciones/_view/{nombre}?group=true"
    return SESSION.get(url, timeout=600).json().get("rows", [])


def ejecutar_agregaciones():
    crear_vistas_agregacion()

    print("--- Ventas por categoría ---")
    ventas = sorted(consultar_vista("ventas_por_categoria"), key=lambda r: r["value"], reverse=True)
    for r in ventas:
        print(f"  {str(r['key']):<18} ${r['value']:>14,.2f}")

    print("\n--- Ticket promedio por estado de pedido ---")
    for r in consultar_vista("ticket_por_estado"):
        v = r["value"]
        print(f"  {r['key']:<12} pedidos: {v['count']:>6} | promedio: ${v['sum'] / v['count']:>9,.2f} "
              f"| mín: ${v['min']:,.2f} | máx: ${v['max']:,.2f}")

    print("\n--- Top 5 productos más vendidos (unidades) ---")
    top = sorted(consultar_vista("productos_mas_vendidos"), key=lambda r: r["value"], reverse=True)[:5]
    for r in top:
        print(f"  {r['key']}  {r['value']} unidades")

    registrar("Req 6: agregaciones MapReduce", bool(ventas and top), "3 vistas consultadas")


ejecutar_agregaciones()

--- Ventas por categoría ---
  Artesanias         $ 15,989,377.14
  Ropa               $ 15,857,943.45
  Hogar              $ 15,617,635.94
  Tecnologia         $ 15,444,370.03
  Cuidado_Personal   $ 15,376,700.93
  Libros             $ 15,356,986.53
  Alimentos          $ 15,080,301.64
  Mascotas           $ 15,034,859.26

--- Ticket promedio por estado de pedido ---
  cancelado    pedidos:  16827 | promedio: $ 3,770.31 | mín: $7.18 | máx: $14,262.16
  completado   pedidos:  16690 | promedio: $ 3,755.05 | mín: $5.70 | máx: $15,846.64
  pendiente    pedidos:  16483 | promedio: $ 3,706.03 | mín: $5.14 | máx: $15,591.81

--- Top 5 productos más vendidos (unidades) ---
  PROD-013242  28 unidades
  PROD-021866  28 unidades
  PROD-014110  26 unidades
  PROD-016627  26 unidades
  PROD-003762  25 unidades
  [OK] Req 6: agregaciones MapReduce  3 vistas consultadas


## Requisito 7 · Evolución del esquema sin migración
Se demuestra con tres acciones sobre la misma base de datos, sin `ALTER TABLE` ni migración:
1. Una **categoría nueva** con atributos nunca vistos.
2. Un **atributo nuevo** en un producto existente.
3. Una **validación** (`validate_doc_update`) que mantiene la consistencia: rechaza documentos inválidos aunque el esquema sea flexible.

In [10]:
def instalar_validacion():
    guardar_design_doc({
        "_id": "_design/validacion",
        "validate_doc_update": (
            "function (newDoc, oldDoc, userCtx) {"
            "  if (newDoc._deleted || newDoc._id.indexOf('_design/') === 0) { return; }"
            "  if (['producto', 'cliente', 'pedido'].indexOf(newDoc.tipo_doc) === -1) {"
            "    throw ({forbidden: 'tipo_doc invalido'}); }"
            "  if (newDoc.tipo_doc === 'producto') {"
            "    if (!newDoc.categoria) { throw ({forbidden: 'categoria requerida'}); }"
            "    if (typeof newDoc.precio !== 'number' || newDoc.precio < 0) {"
            "      throw ({forbidden: 'precio invalido'}); } }"
            "  if (newDoc.tipo_doc === 'pedido' && (!newDoc.lineas_detalle || newDoc.lineas_detalle.length === 0)) {"
            "    throw ({forbidden: 'el pedido necesita lineas'}); }"
            "}"
        ),
    })


def demostrar_evolucion_esquema():
    instalar_validacion()
    print("[1] Validación instalada (consistencia)\n")

    # A) Categoría completamente nueva, con sus propios atributos
    nuevo = {
        "_id": "PROD-999999", "tipo_doc": "producto", "producto_id": "PROD-999999",
        "nombre": "Drone de Monitoreo Agrícola", "categoria": "Tecnologia_Agricola",
        "precio": 2500.00, "stock": 10,
        "atributos": {"autonomia_minutos": 45, "camara_multiespectral": True, "alcance_km": 12.5},
    }
    r = SESSION.put(f"{DB_URL}/PROD-999999", json=nuevo, timeout=30)
    estado = {201: "creada", 202: "creada", 409: "ya existía"}.get(r.status_code, f"error {r.status_code}")
    print(f"[2] Categoría nueva 'Tecnologia_Agricola': {estado}")
    hallados = buscar({"tipo_doc": "producto", "categoria": "Tecnologia_Agricola"},
                      use_index="idx_product_category", titulo="    consulta de la categoría nueva")

    # B) Atributo nuevo en un producto existente
    ropa = SESSION.post(f"{DB_URL}/_find", timeout=60,
                        json={"selector": {"tipo_doc": "producto", "categoria": "Ropa"}, "limit": 1}).json()["docs"][0]
    ropa["atributos"]["certificacion_ecologica"] = True
    r2 = SESSION.put(f"{DB_URL}/{ropa['_id']}", json=ropa, timeout=30)
    print(f"[3] Atributo nuevo 'certificacion_ecologica' en {ropa['producto_id']}: HTTP {r2.status_code}")

    # C) Consistencia: un producto inválido es rechazado por la base de datos
    invalido = {"tipo_doc": "producto", "categoria": "Ropa", "nombre": "Prueba", "precio": -5}
    r3 = SESSION.post(DB_URL, json=invalido, timeout=30)
    print(f"[4] Producto con precio negativo: HTTP {r3.status_code} -> {r3.json().get('reason')}")

    registrar("Req 7: evolución del esquema",
              r.status_code in (201, 202, 409) and bool(hallados.get("docs")) and r3.status_code == 403,
              "categoría nueva, atributo nuevo y validación")


demostrar_evolucion_esquema()

[1] Validación instalada (consistencia)

[2] Categoría nueva 'Tecnologia_Agricola': creada
    consulta de la categoría nueva: 1 resultados | docs examinados: 1 | 16.0 ms
   aviso de CouchDB: _design/idx_product_category was not used because it does not contain a valid index for this query
[3] Atributo nuevo 'certificacion_ecologica' en PROD-000001: HTTP 201
[4] Producto con precio negativo: HTTP 403 -> precio invalido
  [OK] Req 7: evolución del esquema  categoría nueva, atributo nuevo y validación


## Resumen de cumplimiento

In [11]:
print("=== RESUMEN DE REQUISITOS · MERCADOTICO 360 ===")
for requisito, (ok, detalle) in RESULTADOS.items():
    print(f"[{'OK' if ok else 'FALLO'}] {requisito}  {detalle}")
print(f"\n{sum(ok for ok, _ in RESULTADOS.values())}/{len(RESULTADOS)} requisitos verificados")

=== RESUMEN DE REQUISITOS · MERCADOTICO 360 ===
[OK] Req 1-2: diseño y atributos variables  
[OK] Req 3: índices  4 índices
[OK] Req 4: antes/después de índices  27.3x menos documentos examinados
[OK] Req 4: categoría + precio + atributo  611 productos
[OK] Req 5: pedidos y actualización  11 pedidos recuperados
[OK] Req 6: agregaciones MapReduce  3 vistas consultadas
[OK] Req 7: evolución del esquema  categoría nueva, atributo nuevo y validación

7/7 requisitos verificados
